In [1]:
import pandas as pd

url = (
    "https://raw.githubusercontent.com/"
    "priyavenugopalnair-debug/group7-optimizers/"
    "main/data/round_k1.csv"
)

df = pd.read_csv(url)

print("Number of rows:", len(df))
print("Column names:", df.columns.tolist())

df.head()

Number of rows: 200
Column names: ['sample_id', 'x1', 'x2', 'y']


,sample_id,x1,x2,y
0,1,0.286615,-1.121875,-1.383905
1,2,0.738565,1.021845,4.259384
2,3,-2.000594,-1.270712,-8.543204
3,4,0.107272,-0.317633,-0.313451
4,5,-0.039387,-0.897196,-1.912552


In [2]:
# Recalculate the expected answer for every row
calculated_y = 3 * df["x1"] + 2 * df["x2"]

# Compare the provided answers with our calculated answers
difference = df["y"] - calculated_y

print("Largest absolute difference:", difference.abs().max())
print("All 200 targets match:", (difference.abs() < 1e-10).all())

Largest absolute difference: 1.7763568394002505e-15
All 200 targets match: True


In [3]:
A = (df["x1"] ** 2).mean()
B = (df["x1"] * df["x2"]).mean()
C = (df["x2"] ** 2).mean()

print("A — mean(x1 squared):", A)
print("B — mean(x1 times x2):", B)
print("C — mean(x2 squared):", C)

A — mean(x1 squared): 1.0000000000000002
B — mean(x1 times x2): -1.7763568394002505e-17
C — mean(x2 squared): 0.9999999999999997


In [4]:
import numpy as np

H = np.array([
    [A, B],
    [B, C]
])

print("Round dataset Hessian:")
print(H)

print("\nMean of x1:", df["x1"].mean())
print("Mean of x2:", df["x2"].mean())

Round dataset Hessian:
[[ 1.00000000e+00 -1.77635684e-17]
 [-1.77635684e-17  1.00000000e+00]]

Mean of x1: -6.661338147750939e-18
Mean of x2: -6.217248937900877e-17


In [5]:
eigenvalues = np.linalg.eigvalsh(H)

lambda_min = eigenvalues[0]
lambda_max = eigenvalues[-1]
kappa = lambda_max / lambda_min

print("Smallest eigenvalue:", lambda_min)
print("Largest eigenvalue:", lambda_max)
print("Condition number kappa:", kappa)

Smallest eigenvalue: 0.9999999999999997
Largest eigenvalue: 1.0000000000000002
Condition number kappa: 1.0000000000000007


In [6]:
gd_step = 2 / (lambda_min + lambda_max)
upper_boundary = 2 / lambda_max

print("Selected GD step size:", gd_step)
print("Upper stability boundary:", upper_boundary)
print("The GD step must be greater than 0 and below this boundary.")

Selected GD step size: 1.0
Upper stability boundary: 1.9999999999999996
The GD step must be greater than 0 and below this boundary.


In [7]:
base_url = (
    "https://raw.githubusercontent.com/"
    "priyavenugopalnair-debug/group7-optimizers/main/data/"
)

filenames = [
    "round_k1.csv",
    "moderate_k10.csv",
    "narrow_k100.csv"
]

records = []

for filename in filenames:
    data = pd.read_csv(base_url + filename)

    # Use only the two input columns
    X = data[["x1", "x2"]].to_numpy()
    y = data["y"].to_numpy()
    n = len(data)

    # Calculate the Hessian from the actual data
    case_H = X.T @ X / n
    case_eigenvalues = np.linalg.eigvalsh(case_H)

    smallest = case_eigenvalues[0]
    largest = case_eigenvalues[-1]

    # Check the provided targets against the generating rule
    target_difference = y - (3 * X[:, 0] + 2 * X[:, 1])

    records.append({
        "dataset": filename,
        "rows": n,
        "H11": case_H[0, 0],
        "H12": case_H[0, 1],
        "H22": case_H[1, 1],
        "lambda_min": smallest,
        "lambda_max": largest,
        "kappa": largest / smallest,
        "GD_step": 2 / (smallest + largest),
        "mean_x1": X[:, 0].mean(),
        "mean_x2": X[:, 1].mean(),
        "max_target_difference": np.abs(target_difference).max(),
        "targets_match": bool(
            np.all(np.abs(target_difference) < 1e-10)
        )
    })

    print(filename)
    print("Hessian:")
    print(case_H)
    print()

round_k1.csv
Hessian:
[[ 1.0000000e+00 -5.8770986e-17]
 [-5.8770986e-17  1.0000000e+00]]

moderate_k10.csv
Hessian:
[[ 1.00000000e+00 -9.76584418e-18]
 [-9.76584418e-18  1.00000000e-01]]

narrow_k100.csv
Hessian:
[[ 1.00000000e+00 -1.66120714e-18]
 [-1.66120714e-18  1.00000000e-02]]



In [8]:
summary = pd.DataFrame(records)

print("TABLE 1 — Hessians, eigenvalues and GD step sizes")

display(summary[[
    "dataset",
    "H11",
    "H12",
    "H22",
    "lambda_min",
    "lambda_max",
    "kappa",
    "GD_step"
]])

print("\nTABLE 2 — Dataset sanity checks")

display(summary[[
    "dataset",
    "rows",
    "mean_x1",
    "mean_x2",
    "max_target_difference",
    "targets_match"
]])

TABLE 1 — Hessians, eigenvalues and GD step sizes


,dataset,H11,H12,H22,lambda_min,lambda_max,kappa,GD_step
0,round_k1.csv,1.0,-5.877099e-17,1.00,1.00,1.0,1.0,1.000000
1,moderate_k10.csv,1.0,-9.765844e-18,0.10,0.10,1.0,10.0,1.818182
2,narrow_k100.csv,1.0,-1.661207e-18,0.01,0.01,1.0,100.0,1.980198



TABLE 2 — Dataset sanity checks


,dataset,rows,mean_x1,mean_x2,max_target_difference,targets_match
0,round_k1.csv,200,-6.661338e-18,-6.217249e-17,1.776357e-15,True
1,moderate_k10.csv,200,-6.661338e-18,-1.110223e-17,1.776357e-15,True
2,narrow_k100.csv,200,-6.661338e-18,1.110223e-18,1.776357e-15,True


In [9]:
summary.to_csv("varuna_dataset_checks.csv", index=False)

print("Saved: varuna_dataset_checks.csv")

Saved: varuna_dataset_checks.csv


In [10]:
from google.colab import files

files.download("varuna_dataset_checks.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
def evaluate_weights(data, weights):
    X = data[["x1", "x2"]].to_numpy(dtype=np.float64)
    y = data["y"].to_numpy(dtype=np.float64)

    w = np.array(weights, dtype=np.float64)
    true_weights = np.array([3.0, 2.0])

    predictions = X @ w
    errors = predictions - y

    loss = 0.5 * np.mean(errors ** 2)
    gradient = X.T @ errors / len(data)

    return {
        "loss": float(loss),
        "gradient_norm_L2": float(np.linalg.norm(gradient)),
        "parameter_error_L2": float(
            np.linalg.norm(w - true_weights)
        )
    }

print("Metric function ready.")

Metric function ready.


In [12]:
example_settings = [
    ("Starting weights", [0.0, 0.0]),
    ("Illustrative intermediate weights", [2.5, 1.5]),
    ("Known correct weights", [3.0, 2.0])
]

example_rows = []

for label, weights in example_settings:
    example_rows.append({
        "setting": label,
        "w1": weights[0],
        "w2": weights[1],
        **evaluate_weights(df, weights)
    })

metric_examples = pd.DataFrame(example_rows)

print("Metric examples on the round dataset:")
display(metric_examples)

Metric examples on the round dataset:


,setting,w1,w2,loss,gradient_norm_L2,parameter_error_L2
0,Starting weights,0.0,0.0,6.500000e+00,3.605551e+00,3.605551
1,Illustrative intermediate weights,2.5,1.5,2.500000e-01,7.071068e-01,0.707107
2,Known correct weights,3.0,2.0,1.242664e-31,1.442090e-16,0.000000


In [13]:
metric_check_rows = []

for filename in filenames:
    case_data = pd.read_csv(base_url + filename)

    for label, weights in [
        ("Starting weights", [0.0, 0.0]),
        ("Known correct weights", [3.0, 2.0])
    ]:
        metric_check_rows.append({
            "dataset": filename,
            "checkpoint": label,
            **evaluate_weights(case_data, weights)
        })

metric_checks = pd.DataFrame(metric_check_rows)

print("Metric checks — these are not optimizer training results:")
display(metric_checks)

Metric checks — these are not optimizer training results:


,dataset,checkpoint,loss,gradient_norm_L2,parameter_error_L2
0,round_k1.csv,Starting weights,6.500000e+00,3.605551e+00,3.605551
1,round_k1.csv,Known correct weights,1.242664e-31,1.442090e-16,0.000000
2,moderate_k10.csv,Starting weights,4.700000e+00,3.006659e+00,3.605551
3,moderate_k10.csv,Known correct weights,9.984454e-32,4.446578e-17,0.000000
4,narrow_k100.csv,Starting weights,4.520000e+00,3.000067e+00,3.605551
5,narrow_k100.csv,Known correct weights,9.194991e-32,3.001569e-17,0.000000


In [14]:
import math

tolerance = 1e-6
baseline_rows = []

for _, row in summary.iterrows():
    kappa_value = float(row["kappa"])

    # Starting gradient norm for the ideal specified Hessian
    initial_gradient_norm = np.linalg.norm(
        [3.0, 2.0 / kappa_value]
    )

    if np.isclose(kappa_value, 1.0):
        predicted_updates = 1
    else:
        shrink_factor = (
            (kappa_value - 1) / (kappa_value + 1)
        )

        threshold = (
            math.log(tolerance / initial_gradient_norm)
            / math.log(shrink_factor)
        )

        # Smallest integer update count satisfying the strict bound
        predicted_updates = math.floor(threshold) + 1

    baseline_rows.append({
        "dataset": row["dataset"],
        "kappa": kappa_value,
        "lambda_min": row["lambda_min"],
        "lambda_max": row["lambda_max"],
        "GD_step": row["GD_step"],
        "predicted_GD_updates": predicted_updates,
        "status": "Theoretical prediction"
    })

baseline_predictions = pd.DataFrame(baseline_rows)

display(baseline_predictions)

,dataset,kappa,lambda_min,lambda_max,GD_step,predicted_GD_updates,status
0,round_k1.csv,1.0,1.00,1.0,1.000000,1,Theoretical prediction
1,moderate_k10.csv,10.0,0.10,1.0,1.818182,75,Theoretical prediction
2,narrow_k100.csv,100.0,0.01,1.0,1.980198,746,Theoretical prediction


In [15]:
import zipfile
from google.colab import files

summary.to_csv("varuna_dataset_checks.csv", index=False)

metric_checks.to_csv(
    "varuna_metric_checks.csv",
    index=False
)

baseline_predictions.to_csv(
    "varuna_GD_predictions.csv",
    index=False
)

output_files = [
    "varuna_dataset_checks.csv",
    "varuna_metric_checks.csv",
    "varuna_GD_predictions.csv"
]

with zipfile.ZipFile(
    "varuna_slide3_evidence.zip", "w"
) as archive:
    for filename in output_files:
        archive.write(filename)

files.download("varuna_slide3_evidence.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Evaluation metrics

1. Iterations to tolerance
The number of optimizer iterations needed to meet our stopping rule.

2. Gradient evaluations
The number of times the gradient is calculated. L-BFGS may calculate it several times during one iteration.

3. Final loss
The remaining prediction error after training, measured using half the mean squared error.

4. Gradient norm
The Euclidean length of the gradient. Our stopping rule is gradient norm < 0.000001.

5. Parameter error
The Euclidean distance between the learned weights and the known correct weights, (3, 2).

## Worked calculation for the narrow dataset

For narrow_k100.csv, the calculated Hessian is approximately:

H = [[1, 0], [0, 0.01]]

Because this matrix is diagonal, its eigenvalues are its diagonal entries:
1 and 0.01.

Condition number:
kappa = largest eigenvalue / smallest eigenvalue
kappa = 1 / 0.01 = 100

Selected fixed GD step size:
alpha = 2 / (largest eigenvalue + smallest eigenvalue)
alpha = 2 / (1 + 0.01)
alpha ≈ 1.980198

For this quadratic loss, the fixed-step stability condition is:
0 < alpha < 2

The selected step size satisfies this condition.

## Prior work and sources

Kingma, D. P., and Ba, J. (2015).
Adam: A Method for Stochastic Optimization.
International Conference on Learning Representations (ICLR).

Paper: https://arxiv.org/abs/1412.6980

Verified datasets:
- MNIST: Sections 6.1 and 6.2.
- CIFAR-10: Section 6.3.

This paper provides background on optimizer evaluation.
Our experiment compares GD and L-BFGS using our three
AI-assisted synthetic regression datasets.